In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. DATA LOADING
# ============================================================================

# Load your data (adjust file path as needed)
df = pd.read_csv(r"C:\Users\PCexpress\Downloads\predictive_maintenance_dw.csv")

print(f"Initial dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

# ============================================================================
# 2. DATA CLEANING & PREPROCESSING
# ============================================================================

def clean_maintenance_data(df):
    """
    Clean and prepare predictive maintenance data for analysis
    """
    
    print("\n" + "="*80)
    print("DATA CLEANING PROCESS")
    print("="*80)
    
    # --- REMOVE DUPLICATES ---
    initial_rows = len(df)
    df = df.drop_duplicates(subset=['record_id'])
    print(f"\nDuplicates removed: {initial_rows - len(df)}")
    
    # --- DATE/TIME CLEANING ---
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # If timestamp missing but date exists, create timestamp
    df['timestamp'] = df['timestamp'].fillna(df['date'])
    
    # Extract time features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month
    df['year'] = df['timestamp'].dt.year
    df['week'] = df['timestamp'].dt.isocalendar().week
    
    # --- SHIFT CLEANING ---
    df['shift'] = df['shift'].fillna('Unknown')
    
    # --- EQUIPMENT INFORMATION ---
    df['equipment_id'].fillna('UNKNOWN', inplace=True)
    df['equipment_type'].fillna('Unknown', inplace=True)
    df['plant_area'].fillna('Unknown', inplace=True)
    df['manufacturer'].fillna('Unknown', inplace=True)
    
    # --- SENSOR DATA CLEANING ---
    
    print("\nMissing values before sensor cleaning:")
    sensor_cols = ['temperature_c', 'vibration_mm_s', 'pressure_bar', 'rpm', 
                   'load_percent', 'efficiency_percent', 'energy_consumption_kwh']
    print(df[sensor_cols].isnull().sum())
    
    # Temperature (typical range: -50 to 200°C)
    df['temperature_c'] = pd.to_numeric(df['temperature_c'], errors='coerce')
    df['temperature_c'] = df['temperature_c'].clip(lower=-50, upper=200)
    df['temperature_c'] = df.groupby('equipment_type')['temperature_c'].transform(
        lambda x: x.fillna(x.median())
    )
    df['temperature_c'].fillna(df['temperature_c'].median(), inplace=True)
    
    # Vibration (typical range: 0 to 100 mm/s)
    df['vibration_mm_s'] = pd.to_numeric(df['vibration_mm_s'], errors='coerce')
    df['vibration_mm_s'] = df['vibration_mm_s'].clip(lower=0, upper=100)
    df['vibration_mm_s'] = df.groupby('equipment_type')['vibration_mm_s'].transform(
        lambda x: x.fillna(x.median())
    )
    df['vibration_mm_s'].fillna(df['vibration_mm_s'].median(), inplace=True)
    
    # Pressure (typical range: 0 to 500 bar)
    df['pressure_bar'] = pd.to_numeric(df['pressure_bar'], errors='coerce')
    df['pressure_bar'] = df['pressure_bar'].clip(lower=0, upper=500)
    df['pressure_bar'] = df.groupby('equipment_type')['pressure_bar'].transform(
        lambda x: x.fillna(x.median())
    )
    df['pressure_bar'].fillna(df['pressure_bar'].median(), inplace=True)
    
    # RPM (typical range: 0 to 10000)
    df['rpm'] = pd.to_numeric(df['rpm'], errors='coerce')
    df['rpm'] = df['rpm'].clip(lower=0, upper=10000)
    df['rpm'] = df.groupby('equipment_type')['rpm'].transform(
        lambda x: x.fillna(x.median())
    )
    df['rpm'].fillna(df['rpm'].median(), inplace=True)
    
    # Load Percent (0 to 100)
    df['load_percent'] = pd.to_numeric(df['load_percent'], errors='coerce')
    df['load_percent'] = df['load_percent'].clip(lower=0, upper=100)
    df['load_percent'].fillna(df['load_percent'].median(), inplace=True)
    
    # Efficiency Percent (0 to 100)
    df['efficiency_percent'] = pd.to_numeric(df['efficiency_percent'], errors='coerce')
    df['efficiency_percent'] = df['efficiency_percent'].clip(lower=0, upper=100)
    df['efficiency_percent'].fillna(df['efficiency_percent'].median(), inplace=True)
    
    # Energy Consumption (non-negative)
    df['energy_consumption_kwh'] = pd.to_numeric(df['energy_consumption_kwh'], errors='coerce')
    df['energy_consumption_kwh'] = df['energy_consumption_kwh'].clip(lower=0)
    df['energy_consumption_kwh'].fillna(df['energy_consumption_kwh'].median(), inplace=True)
    
    # --- OPERATING HOURS ---
    df['operating_hours'] = pd.to_numeric(df['operating_hours'], errors='coerce')
    df['operating_hours'] = df['operating_hours'].clip(lower=0, upper=100000)
    df['operating_hours'].fillna(0, inplace=True)
    
    # --- MAINTENANCE DATA ---
    df['maintenance_type'].fillna('None', inplace=True)
    
    df['days_since_last_maintenance'] = pd.to_numeric(
        df['days_since_last_maintenance'], errors='coerce'
    )
    df['days_since_last_maintenance'] = df['days_since_last_maintenance'].clip(lower=0, upper=3650)
    df['days_since_last_maintenance'].fillna(
        df['days_since_last_maintenance'].median(), inplace=True
    )
    
    # --- FAILURE DATA ---
    
    # Clean failure flag
    df['failure_flag'] = df['failure_flag'].map({
        1: 1, '1': 1, 'Yes': 1, 'yes': 1, 'Y': 1, 'y': 1, True: 1,
        0: 0, '0': 0, 'No': 0, 'no': 0, 'N': 0, 'n': 0, False: 0
    }).fillna(0).astype(int)
    
    # Failure type
    df['failure_type'].fillna('No Failure', inplace=True)
    df.loc[df['failure_flag'] == 0, 'failure_type'] = 'No Failure'
    
    # Downtime
    df['downtime_minutes'] = pd.to_numeric(df['downtime_minutes'], errors='coerce')
    df['downtime_minutes'] = df['downtime_minutes'].clip(lower=0, upper=10080)  # Max 1 week
    df['downtime_minutes'].fillna(0, inplace=True)
    
    # If failure flag is 1 but downtime is 0, set minimum downtime
    df.loc[(df['failure_flag'] == 1) & (df['downtime_minutes'] == 0), 'downtime_minutes'] = 30
    
    # --- RISK LEVEL ---
    df['failure_risk_level'].fillna('Low', inplace=True)
    
    # Standardize risk levels
    risk_mapping = {
        'low': 'Low', 'Low': 'Low', 'LOW': 'Low',
        'medium': 'Medium', 'Medium': 'Medium', 'MEDIUM': 'Medium', 'Med': 'Medium',
        'high': 'High', 'High': 'High', 'HIGH': 'High',
        'critical': 'Critical', 'Critical': 'Critical', 'CRITICAL': 'Critical'
    }
    df['failure_risk_level'] = df['failure_risk_level'].map(risk_mapping).fillna('Low')
    
    print("\nMissing values after cleaning:")
    print(df.isnull().sum())
    
    print(f"\nFinal dataset shape: {df.shape}")
    
    return df


# ============================================================================
# 3. FEATURE ENGINEERING
# ============================================================================

def engineer_features(df):
    """
    Create derived features for better analysis
    """
    
    print("\n" + "="*80)
    print("FEATURE ENGINEERING")
    print("="*80)
    
    # --- ANOMALY DETECTION FEATURES ---
    
    # Temperature anomaly (z-score by equipment type)
    df['temp_zscore'] = df.groupby('equipment_type')['temperature_c'].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
    )
    df['temp_anomaly'] = (abs(df['temp_zscore']) > 2).astype(int)
    
    # Vibration anomaly
    df['vibration_zscore'] = df.groupby('equipment_type')['vibration_mm_s'].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
    )
    df['vibration_anomaly'] = (abs(df['vibration_zscore']) > 2).astype(int)
    
    # --- HEALTH INDEX (0-100 scale, 100 = healthy) ---
    
    # Normalize each parameter (0-1 scale where 1 is worst)
    df['temp_normalized'] = (df['temperature_c'] - df['temperature_c'].min()) / \
                            (df['temperature_c'].max() - df['temperature_c'].min())
    
    df['vibration_normalized'] = (df['vibration_mm_s'] - df['vibration_mm_s'].min()) / \
                                 (df['vibration_mm_s'].max() - df['vibration_mm_s'].min())
    
    df['efficiency_normalized'] = 1 - (df['efficiency_percent'] / 100)  # Inverse for efficiency
    
    df['load_normalized'] = df['load_percent'] / 100
    
    # Calculate health index (weighted average, inverted to 0-100)
    df['health_index'] = 100 - (
        (df['temp_normalized'] * 0.25 +
         df['vibration_normalized'] * 0.30 +
         df['efficiency_normalized'] * 0.25 +
         df['load_normalized'] * 0.20) * 100
    )
    df['health_index'] = df['health_index'].clip(lower=0, upper=100)
    
    # --- FAILURE PROBABILITY (based on multiple factors) ---
    
    # Risk level numeric
    risk_numeric = {'Low': 0.1, 'Medium': 0.4, 'High': 0.7, 'Critical': 0.95}
    df['risk_numeric'] = df['failure_risk_level'].map(risk_numeric)
    
    # Failure probability based on health index and risk level
    df['failure_probability'] = (
        (100 - df['health_index']) / 100 * 0.5 +  # Health contribution
        df['risk_numeric'] * 0.3 +                 # Risk level contribution
        (df['days_since_last_maintenance'] / df['days_since_last_maintenance'].max()) * 0.2
    ) * 100
    
    df['failure_probability'] = df['failure_probability'].clip(lower=0, upper=100)
    
    # --- MAINTENANCE URGENCY ---
    df['maintenance_urgency'] = pd.cut(
        df['failure_probability'],
        bins=[0, 25, 50, 75, 100],
        labels=['Low', 'Medium', 'High', 'Critical']
    )
    
    # --- HIGH RISK FLAG ---
    df['high_risk_flag'] = (
        (df['failure_risk_level'].isin(['High', 'Critical'])) |
        (df['failure_probability'] > 70) |
        (df['health_index'] < 40)
    ).astype(int)
    
    # --- OPERATING CONDITION ---
    df['operating_condition'] = pd.cut(
        df['health_index'],
        bins=[0, 40, 60, 80, 100],
        labels=['Critical', 'Poor', 'Fair', 'Good']
    )
    
    # --- DOWNTIME HOURS ---
    df['downtime_hours'] = df['downtime_minutes'] / 60
    
    print("✅ Features engineered successfully")
    
    return df


# ============================================================================
# 4. OVERALL KPIs
# ============================================================================

def calculate_overall_kpis(df):
    """
    Calculate high-level predictive maintenance KPIs
    """
    
    kpis = {}
    
    # 1. Failure Rate %
    total_records = len(df)
    total_failures = df['failure_flag'].sum()
    kpis['failure_rate_pct'] = (total_failures / total_records * 100) if total_records > 0 else 0
    
    # 2. MTBF (Mean Time Between Failures) in hours
    # Group by equipment and calculate time between failures
    mtbf_list = []
    for equipment in df['equipment_id'].unique():
        eq_data = df[df['equipment_id'] == equipment].sort_values('timestamp')
        failure_times = eq_data[eq_data['failure_flag'] == 1]['timestamp']
        
        if len(failure_times) > 1:
            time_diffs = failure_times.diff().dropna()
            mtbf_hours = time_diffs.dt.total_seconds() / 3600
            mtbf_list.extend(mtbf_hours.tolist())
    
    kpis['mtbf_hours'] = np.mean(mtbf_list) if mtbf_list else 0
    
    # 3. MTTR (Mean Time To Repair) in hours
    failure_records = df[df['failure_flag'] == 1]
    kpis['mttr_hours'] = failure_records['downtime_hours'].mean() if len(failure_records) > 0 else 0
    
    # 4. High Risk Count
    kpis['high_risk_count'] = df['high_risk_flag'].sum()
    kpis['high_risk_equipment_count'] = df[df['high_risk_flag'] == 1]['equipment_id'].nunique()
    
    # 5. Average Temperature
    kpis['avg_temperature_c'] = df['temperature_c'].mean()
    
    # 6. Average Vibration
    kpis['avg_vibration_mm_s'] = df['vibration_mm_s'].mean()
    
    # 7. Overall Health Index
    kpis['avg_health_index'] = df['health_index'].mean()
    
    # 8. Average Failure Probability %
    kpis['avg_failure_probability_pct'] = df['failure_probability'].mean()
    
    # 9. Total Downtime
    kpis['total_downtime_hours'] = df['downtime_hours'].sum()
    kpis['total_downtime_minutes'] = df['downtime_minutes'].sum()
    
    # 10. Equipment Availability %
    total_operating_hours = df['operating_hours'].sum()
    total_downtime_hours = df['downtime_hours'].sum()
    kpis['availability_pct'] = (
        (total_operating_hours - total_downtime_hours) / total_operating_hours * 100
    ) if total_operating_hours > 0 else 100
    
    # 11. Additional metrics
    kpis['total_equipment'] = df['equipment_id'].nunique()
    kpis['failed_equipment'] = df[df['failure_flag'] == 1]['equipment_id'].nunique()
    kpis['avg_days_since_maintenance'] = df['days_since_last_maintenance'].mean()
    
    return pd.Series(kpis)


# ============================================================================
# 5. RISK LEVEL DISTRIBUTION
# ============================================================================

def calculate_risk_distribution(df):
    """
    Analyze risk level distribution
    """
    
    risk_dist = df.groupby('failure_risk_level').agg({
        'record_id': 'count',
        'equipment_id': 'nunique',
        'failure_flag': 'sum',
        'downtime_hours': 'sum',
        'failure_probability': 'mean',
        'health_index': 'mean'
    }).reset_index()
    
    risk_dist.columns = [
        'risk_level', 'record_count', 'equipment_count', 
        'failure_count', 'total_downtime_hours', 
        'avg_failure_probability', 'avg_health_index'
    ]
    
    # Calculate percentages
    total_records = risk_dist['record_count'].sum()
    risk_dist['percentage'] = (risk_dist['record_count'] / total_records * 100)
    
    # Sort by severity
    risk_order = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
    risk_dist['sort_order'] = risk_dist['risk_level'].map(risk_order)
    risk_dist = risk_dist.sort_values('sort_order').drop('sort_order', axis=1)
    
    return risk_dist


# ============================================================================
# 6. DOWNTIME ANALYSIS
# ============================================================================

def calculate_downtime_per_equipment(df):
    """
    Calculate downtime metrics per equipment
    """
    
    downtime_analysis = df.groupby(['equipment_id', 'equipment_type']).agg({
        'downtime_minutes': 'sum',
        'downtime_hours': 'sum',
        'failure_flag': 'sum',
        'operating_hours': 'max',
        'health_index': 'mean',
        'failure_probability': 'mean',
        'high_risk_flag': 'sum',
        'record_id': 'count'
    }).reset_index()
    
    downtime_analysis.columns = [
        'equipment_id', 'equipment_type', 'total_downtime_minutes',
        'total_downtime_hours', 'failure_count', 'total_operating_hours',
        'avg_health_index', 'avg_failure_probability', 'high_risk_count',
        'measurement_count'
    ]
    
    # Calculate availability
    downtime_analysis['availability_pct'] = (
        (downtime_analysis['total_operating_hours'] - downtime_analysis['total_downtime_hours']) /
        downtime_analysis['total_operating_hours'] * 100
    ).clip(lower=0, upper=100)
    
    # Downtime per failure
    downtime_analysis['avg_downtime_per_failure'] = (
        downtime_analysis['total_downtime_hours'] / downtime_analysis['failure_count']
    ).fillna(0)
    
    # Sort by total downtime
    downtime_analysis = downtime_analysis.sort_values('total_downtime_hours', ascending=False)
    
    return downtime_analysis


# ============================================================================
# 7. EQUIPMENT HEALTH ANALYSIS
# ============================================================================

def calculate_equipment_health(df):
    """
    Detailed health analysis per equipment
    """
    
    health_analysis = df.groupby(['equipment_id', 'equipment_type', 'plant_area']).agg({
        'health_index': 'mean',
        'failure_probability': 'mean',
        'temperature_c': ['mean', 'max'],
        'vibration_mm_s': ['mean', 'max'],
        'pressure_bar': 'mean',
        'efficiency_percent': 'mean',
        'operating_hours': 'max',
        'days_since_last_maintenance': 'mean',
        'failure_flag': 'sum',
        'high_risk_flag': 'sum',
        'record_id': 'count'
    }).reset_index()
    
    health_analysis.columns = [
        'equipment_id', 'equipment_type', 'plant_area',
        'avg_health_index', 'avg_failure_probability',
        'avg_temperature', 'max_temperature',
        'avg_vibration', 'max_vibration',
        'avg_pressure', 'avg_efficiency',
        'total_operating_hours', 'avg_days_since_maintenance',
        'failure_count', 'high_risk_count', 'measurement_count'
    ]
    
    # Health status classification
    health_analysis['health_status'] = pd.cut(
        health_analysis['avg_health_index'],
        bins=[0, 40, 60, 80, 100],
        labels=['Critical', 'Poor', 'Fair', 'Good']
    )
    
    # Priority ranking (lower health index = higher priority)
    health_analysis['maintenance_priority'] = health_analysis['avg_health_index'].rank()
    
    # Sort by health index (worst first)
    health_analysis = health_analysis.sort_values('avg_health_index')
    
    return health_analysis


# ============================================================================
# 8. FAILURE TYPE ANALYSIS
# ============================================================================

def calculate_failure_analysis(df):
    """
    Analyze failure patterns by type
    """
    
    failure_data = df[df['failure_flag'] == 1]
    
    if len(failure_data) == 0:
        print("No failures recorded in dataset")
        return pd.DataFrame()
    
    failure_analysis = failure_data.groupby('failure_type').agg({
        'record_id': 'count',
        'equipment_id': 'nunique',
        'downtime_hours': ['sum', 'mean', 'max'],
        'temperature_c': 'mean',
        'vibration_mm_s': 'mean',
        'load_percent': 'mean',
        'operating_hours': 'mean'
    }).reset_index()
    
    failure_analysis.columns = [
        'failure_type', 'occurrence_count', 'affected_equipment',
        'total_downtime_hours', 'avg_downtime_hours', 'max_downtime_hours',
        'avg_temperature', 'avg_vibration', 'avg_load', 'avg_operating_hours'
    ]
    
    # Calculate percentage of total failures
    total_failures = failure_analysis['occurrence_count'].sum()
    failure_analysis['percentage'] = (
        failure_analysis['occurrence_count'] / total_failures * 100
    )
    
    # Sort by occurrence
    failure_analysis = failure_analysis.sort_values('occurrence_count', ascending=False)
    
    return failure_analysis


# ============================================================================
# 9. TIME-BASED TRENDS
# ============================================================================

def calculate_monthly_trends(df):
    """
    Calculate KPI trends over time
    """
    
    monthly = df.groupby(['year', 'month']).agg({
        'failure_flag': 'sum',
        'record_id': 'count',
        'health_index': 'mean',
        'failure_probability': 'mean',
        'temperature_c': 'mean',
        'vibration_mm_s': 'mean',
        'efficiency_percent': 'mean',
        'downtime_hours': 'sum',
        'high_risk_flag': 'sum',
        'energy_consumption_kwh': 'sum'
    }).reset_index()
    
    monthly.columns = [
        'year', 'month', 'failure_count', 'total_records',
        'avg_health_index', 'avg_failure_probability',
        'avg_temperature', 'avg_vibration', 'avg_efficiency',
        'total_downtime_hours', 'high_risk_count',
        'total_energy_consumption'
    ]
    
    # Calculate failure rate
    monthly['failure_rate_pct'] = (
        monthly['failure_count'] / monthly['total_records'] * 100
    )
    
    # Sort by date
    monthly = monthly.sort_values(['year', 'month'])
    
    return monthly


# ============================================================================
# 10. EQUIPMENT TYPE COMPARISON
# ============================================================================

def calculate_equipment_type_kpis(df):
    """
    Compare KPIs across equipment types
    """
    
    type_kpis = df.groupby('equipment_type').agg({
        'equipment_id': 'nunique',
        'failure_flag': ['sum', 'mean'],
        'health_index': 'mean',
        'failure_probability': 'mean',
        'downtime_hours': 'sum',
        'temperature_c': 'mean',
        'vibration_mm_s': 'mean',
        'efficiency_percent': 'mean',
        'operating_hours': 'sum',
        'high_risk_flag': 'sum',
        'record_id': 'count'
    }).reset_index()
    
    type_kpis.columns = [
        'equipment_type', 'equipment_count',
        'failure_count', 'failure_rate',
        'avg_health_index', 'avg_failure_probability',
        'total_downtime_hours', 'avg_temperature',
        'avg_vibration', 'avg_efficiency',
        'total_operating_hours', 'high_risk_count',
        'measurement_count'
    ]
    
    # Convert failure rate to percentage
    type_kpis['failure_rate_pct'] = type_kpis['failure_rate'] * 100
    
    # Calculate MTBF per equipment type
    mtbf_by_type = []
    for eq_type in df['equipment_type'].unique():
        type_data = df[df['equipment_type'] == eq_type]
        
        mtbf_list = []
        for equipment in type_data['equipment_id'].unique():
            eq_data = type_data[type_data['equipment_id'] == equipment].sort_values('timestamp')
            failure_times = eq_data[eq_data['failure_flag'] == 1]['timestamp']
            
            if len(failure_times) > 1:
                time_diffs = failure_times.diff().dropna()
                mtbf_hours = time_diffs.dt.total_seconds() / 3600
                mtbf_list.extend(mtbf_hours.tolist())
        
        avg_mtbf = np.mean(mtbf_list) if mtbf_list else 0
        mtbf_by_type.append({'equipment_type': eq_type, 'mtbf_hours': avg_mtbf})
    
    mtbf_df = pd.DataFrame(mtbf_by_type)
    type_kpis = type_kpis.merge(mtbf_df, on='equipment_type', how='left')
    
    # Sort by failure rate
    type_kpis = type_kpis.sort_values('failure_rate_pct', ascending=False)
    
    return type_kpis


# ============================================================================
# 11. PREDICTIVE ALERTS
# ============================================================================

def generate_predictive_alerts(df):
    """
    Generate alerts for equipment requiring immediate attention
    """
    
    # Critical alerts
    critical_equipment = df[
        (df['high_risk_flag'] == 1) |
        (df['health_index'] < 40) |
        (df['failure_probability'] > 80)
    ].copy()
    
    if len(critical_equipment) == 0:
        print("No critical alerts")
        return pd.DataFrame()
    
    alerts = critical_equipment.groupby(['equipment_id', 'equipment_type', 'plant_area']).agg({
        'health_index': 'min',
        'failure_probability': 'max',
        'temperature_c': 'max',
        'vibration_mm_s': 'max',
        'days_since_last_maintenance': 'max',
        'failure_risk_level': lambda x: x.mode()[0] if len(x) > 0 else 'Unknown',
        'timestamp': 'max'
    }).reset_index()
    
    alerts.columns = [
        'equipment_id', 'equipment_type', 'plant_area',
        'current_health_index', 'max_failure_probability',
        'max_temperature', 'max_vibration',
        'days_since_maintenance', 'risk_level', 'last_reading'
    ]
    
    # Assign alert severity
    def assign_severity(row):
        if row['current_health_index'] < 30 or row['max_failure_probability'] > 90:
            return 'CRITICAL'
        elif row['current_health_index'] < 50 or row['max_failure_probability'] > 70:
            return 'HIGH'
        else:
            return 'MEDIUM'
    
    alerts['alert_severity'] = alerts.apply(assign_severity, axis=1)
    
    # Sort by severity and health
    severity_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2}
    alerts['sort_order'] = alerts['alert_severity'].map(severity_order)
    alerts = alerts.sort_values(['sort_order', 'current_health_index'])
    alerts = alerts.drop('sort_order', axis=1)
    
    return alerts


# ============================================================================
# 12. MAIN EXECUTION
# ============================================================================

# Clean data
df_cleaned = clean_maintenance_data(df)

# Engineer features
df_enhanced = engineer_features(df_cleaned)

print("\n" + "="*80)
print("OVERALL PREDICTIVE MAINTENANCE KPIs")
print("="*80)
overall_kpis = calculate_overall_kpis(df_enhanced)
print(overall_kpis)

print("\n" + "="*80)
print("RISK LEVEL DISTRIBUTION")
print("="*80)
risk_distribution = calculate_risk_distribution(df_enhanced)
print(risk_distribution)

print("\n" + "="*80)
print("DOWNTIME PER EQUIPMENT (Top 10)")
print("="*80)
downtime_per_equipment = calculate_downtime_per_equipment(df_enhanced)
print(downtime_per_equipment.head(10))

print("\n" + "="*80)
print("EQUIPMENT HEALTH ANALYSIS (Top 10 Critical)")
print("="*80)
equipment_health = calculate_equipment_health(df_enhanced)
print(equipment_health.head(10))

print("\n" + "="*80)
print("FAILURE TYPE ANALYSIS")
print("="*80)
failure_analysis = calculate_failure_analysis(df_enhanced)
if not failure_analysis.empty:
    print(failure_analysis)
else:
    print("No failures to analyze")

print("\n" + "="*80)
print("EQUIPMENT TYPE KPIs")
print("="*80)
equipment_type_kpis = calculate_equipment_type_kpis(df_enhanced)
print(equipment_type_kpis)

print("\n" + "="*80)
print("MONTHLY TRENDS (Last 12 Months)")
print("="*80)
monthly_trends = calculate_monthly_trends(df_enhanced)
print(monthly_trends.tail(12))

print("\n" + "="*80)
print("CRITICAL ALERTS - EQUIPMENT REQUIRING IMMEDIATE ATTENTION")
print("="*80)
predictive_alerts = generate_predictive_alerts(df_enhanced)
if not predictive_alerts.empty:
    print(predictive_alerts.head(20))
else:
    print("No critical alerts")



Initial dataset shape: (600, 22)

First few rows:
   record_id            timestamp        date  shift  equipment_id  \
0          1  2024-01-01 00:00:00  2024-01-01  Night            22   
1          2  2024-01-01 01:00:00  2024-01-01  Night            10   
2          3  2024-01-01 02:00:00  2024-01-01  Night            15   
3          4  2024-01-01 03:00:00  2024-01-01  Night             4   
4          5  2024-01-01 04:00:00  2024-01-01  Night            24   

  equipment_type  plant_area  manufacturer  temperature_c  vibration_mm_s  \
0           Pump   Utilities  Schlumberger          78.97            3.83   
1     Compressor   Utilities           ABB          87.18            3.30   
2     Compressor  Production           ABB          95.48            4.47   
3          Motor    Drilling  Schlumberger          70.88            2.57   
4           Pump    Drilling           ABB          77.11            5.97   

   ...  load_percent  operating_hours  efficiency_percent  \
0  ..